## Tools

Models can request to call tools that perform task sicj as fetchinh data from the web, performing calculations, or accessing a database.

Tools are pairing of :

1.A schema, which defines the input and output of the tool, and

2.The name of the tool, which is used to identify the tool when calling it.

3.description of the tool, which is used to provide additional information about the tool to the model.

4.argument definitions(often a JSON schema) that specify the input parameters for the tool, including their types, constraints, and any required fields. This allows the model to understand how to interact with the tool and what kind of data it can expect in return.

5.function or coroutine to execute the tool's functionality based on the provided arguments. This function can be implemented in Python or any other programming language, and it should handle the logic for processing the input and generating the output according to the tool's schema.

In [14]:
from email import message

from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv()

# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(model="groq:openai/gpt-oss-120b")
response = model.invoke("how does tools work with LLMs and agents?")
response.content

'## Tools, LLMs, and Agents – A Practical Overview  \n\nLarge Language Models (LLMs) are great at **reasoning with text**, but they can’t directly interact with the outside world: they can’t read a file, hit a web API, or run code.  \n**Tools** (sometimes called *functions*, *plugins*, or *external services*) bridge that gap. When an LLM is embedded inside an **agent**—a loop that decides what to do next—it can **invoke** these tools, get concrete results, and feed them back into its next reasoning step.\n\nBelow is a step‑by‑step description of how this works, the common patterns you’ll see in today’s frameworks, and a few concrete code snippets to illustrate the flow.\n\n---\n\n## 1. Core Concepts\n\n| Term | What it means | Example |\n|------|---------------|---------|\n| **LLM** | A generative model that produces text (or tokens) from a prompt. | GPT‑4, Claude, LLaMA 2 |\n| **Tool / Function** | A deterministic piece of code or external API that the LLM can call. It has a **name**,

## creating a tool

In [17]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    return f"The current weather in {city} is sunny and 25°C."

### binding the tool with the model

In [18]:
model_with_tools = model.bind_tools([get_weather])

### call the tool from the model

In [21]:
response = model_with_tools.invoke("What is the weather like in New York?")

print(response)

for tool_call in response.tool_calls:
    #viw tool call made by the model
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What is the weather like in New York?" We need to get current weather. Use the provided function get_weather with city "New York". Then respond with the weather.', 'tool_calls': [{'id': 'fc_51b8b169-7e3a-4444-a253-d3511a4e8e8e', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 131, 'total_tokens': 199, 'completion_time': 0.143024496, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.006571668, 'prompt_tokens_details': None, 'queue_time': 0.317844321, 'total_time': 0.149596164}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1bb47-1d27-7346-b87b-6a15d6afee97-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 

### Tool execution Loop technique

In [22]:
# step 1: model generates a tool call

messages = [{"role": "user", "content": "What is the weather like in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


# step 2: execute the tool call and get the result
for tool_call in ai_msg.tool_calls:
    # execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: pass result back to the model for further processing
final_response = model_with_tools.invoke(messages)
print(final_response)




content='The current weather in New\u202fYork is sunny with a temperature of about **25\u202f°C**. Enjoy the pleasant day!' additional_kwargs={'reasoning_content': 'We have the weather info. Need to answer user.'} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 170, 'total_tokens': 216, 'completion_time': 0.096049312, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.007813098, 'prompt_tokens_details': None, 'queue_time': 0.31783874, 'total_time': 0.10386241}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1bb4d-e8d1-7321-8d26-18c2cf4c1931-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 170, 'output_tokens': 46, 'total_tokens': 216, 'output_token_details': {'reasoning': 12}}


In [23]:
messages

[{'role': 'user', 'content': 'What is the weather like in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks "What is the weather like in New York?" We need to fetch current weather. Use function get_weather with city "New York".', 'tool_calls': [{'id': 'fc_8fd33757-404c-4099-bdaf-7af3a53b2c8f', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 131, 'total_tokens': 189, 'completion_time': 0.122960156, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.005296559, 'prompt_tokens_details': None, 'queue_time': 0.286886128, 'total_time': 0.128256715}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_96d96a151c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1bb4d-e6f1-760a-92ed-51a040c89180-0', tool_calls=[{'name': 'get_weather